In [1]:
"function (and parameter space) definitions for hyperband"
"regression with Keras (multilayer perceptron)"

import numpy as np
import matplotlib.pyplot as plt
import pickle

from hyperopt import hp
from hyperopt.pyll.stochastic import sample

from math import log, sqrt
from time import time
from pprint import pprint

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score as AUC, log_loss, accuracy_score as accuracy
from sklearn.metrics import mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2, explained_variance_score as EVS
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler

from keras.models import Sequential
from keras.layers.core import Dense, Dropout
from keras.layers.normalization import BatchNormalization as BatchNorm
from keras.callbacks import EarlyStopping
from keras.layers.advanced_activations import *

%load_ext autoreload
%autoreload 2

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 8)

/home/bulent/anaconda3/lib/python3.6/site-packages/h5py/__init__.py:34: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


In [2]:
with open('stations-6to31.pkl', 'rb') as f:
    alldata = pickle.load(f)
    
x_train = alldata['x_train']
y_train = alldata['y_train']
x_dev = alldata['x_dev']
y_dev = alldata['y_dev']

print(x_train[0])

[41.04        0.1         6.47083333  9.09995719 13.13619327]


In [3]:
# handle floats which should be integers
# works with flat params
def handle_integers( params ):
    new_params = {}
    for k, v in params.items():
        if type( v ) == float and int( v ) == v:
            new_params[k] = int( v )
        else:
            new_params[k] = v
    
    return new_params

In [8]:
# Only parametric ReLU activations, truncated space based on previous results...

max_layers = 4
max_layer_size = 20

space = {
    'scaler': hp.choice( 's', ( None, 'StandardScaler', 'RobustScaler', 'MinMaxScaler', 'MaxAbsScaler' )),
    'n_layers': hp.quniform( 'ls', 1, max_layers, 1 ),
    'init': hp.choice( 'i', ( 'uniform', 'normal', 'glorot_uniform', 'glorot_normal', 'he_uniform', 'he_normal' )),
    'batch_size': hp.choice( 'bs', ( 16, 32, 64, 128 )),
    'loss': hp.choice( 'l', ( 'mean_absolute_error', 'mean_squared_error' )),
    'learning_rate': hp.choice('lr', (1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7)),
    'momentum': hp.choice('m', (0.2, 0.4, 0.6, 0.8, 0.9)),
    'decay': hp.choice('d', (0, 1, 2, 3, 4)),
    'epochs': hp.choice('e', ())
}

# for each hidden layer, we choose size and extras individually
for i in range( 1, max_layers + 1 ):
    space[ 'layer_{}_size'.format( i )] = hp.quniform( 'ls{}'.format( i ), 
        10, max_layer_size, 1 )
    space[ 'layer_{}_extras'.format( i )] = hp.choice( 'e{}'.format( i ), ( 
        { 'name': 'dropout', 'rate': hp.uniform( 'd{}'.format( i ), 0.1, 0.5, 0.8 )}, 
        { 'name': 'batchnorm' },
        { 'name': None } ))    
    
def get_params():
    params = sample( space )
    return handle_integers( params )

# print hidden layers config in readable way
def print_layers( params ):
    for i in range( 1, params['n_layers'] + 1 ):
        print("layer {} | size: {:>3} | extras: {}".format( i,
            params['layer_{}_size'.format( i )], 
            params['layer_{}_extras'.format( i )]['name'] ))
        if params['layer_{}_extras'.format( i )]['name'] == 'dropout':
            print("- rate: {:.1%}".format( params['layer_{}_extras'.format( i )]['rate'] ))

def print_params( params ):
    pprint({ k: v for k, v in params.items() if not k.startswith( 'layer_' )})
    print_layers( params )

def try_params( n_iterations, params ):
    
    print("iterations:", n_iterations)
    print_params( params )
    
    y_test = y_dev
    if params['scaler']:
        scaler = eval( "{}()".format( params['scaler'] ))
        x_train_ = scaler.fit_transform( x_train.astype( float ))
        x_test_ = scaler.transform( x_dev.astype( float ))
    else:
        x_train_ = x_train
        x_test_ = x_dev
    
    input_dim = x_train_.shape[1]

    model = Sequential()
    model.add( Dense( params['layer_1_size'], kernel_initializer = params['init'], 
        input_dim = input_dim ))
    model.add( PReLU( alpha_initializer=params['init'] ))
    
    for i in range( int( params['n_layers'] ) - 1 ):
        
        extras = 'layer_{}_extras'.format( i + 1 )
        
        if params[extras]['name'] == 'dropout':
            model.add( Dropout( params[extras]['rate'] ))
        elif params[extras]['name'] == 'batchnorm':
            model.add( BatchNorm())
            
        model.add( Dense( params['layer_{}_size'.format( i + 2 )], kernel_initializer = params['init'] ))
        model.add( PReLU( alpha_initializer=params['init'] ))
           
    model.add( Dense( 1, kernel_initializer = params['init'], activation = 'linear' ))

    model.compile( optimizer = params['optimizer'], loss = params['loss'] )
    
    validation_data = ( x_test_, y_test )

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    
    history = model.fit( x_train_, y_train,
        epochs = int( round( n_iterations )),
        batch_size = params['batch_size'], 
        shuffle = params['shuffle'], 
        validation_data = validation_data, 
        callbacks = [ early_stopping ])    
    
    p = model.predict( x_train_, batch_size = params['batch_size'] )

    mse = MSE( y_train, p )
    rmse = sqrt( mse )
    mae = MAE( y_train, p )
    r2 = R2( y_train, p )
    evs = EVS( y_train, p )

    print("\n# training | RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}".format( rmse, mae, r2, evs ))

    p = model.predict( x_test_, batch_size = params['batch_size'] )
    
    mse = MSE( y_test, p )
    rmse = sqrt( mse )
    mae = MAE( y_test, p )
    r2 = R2( y_test, p )
    evs = EVS( y_test, p )

    print("\n# test | RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}".format( rmse, mae, r2, evs ))
    
    return { 'loss': rmse, 'rmse': rmse, 'mae': mae, 'r2':r2, 'evs':evs, 'early_stop': model.stop_training }


In [9]:
from hyperband import Hyperband

output_file = 'hyperband_31stations.pkl'
print("Will save results to", output_file)

hb = Hyperband( get_params, try_params )
results = hb.run( skip_last = 1 )

print("{} total, best:\n".format( len( results )))

for r in sorted( results, key = lambda x: x['loss'] )[:5]:
    print("loss: {:.2%} | {} seconds | {:.1f} iterations | run {} ".format( 
        r['loss'], r['seconds'], r['iterations'], r['counter'] ))
    pprint( r['params'] )

print("saving...")

with open( output_file, 'wb' ) as f:
    pickle.dump( results, f )

Will save results to hyperband_31stations.pkl

*** 81 configurations x 1.0 iterations each

1 | Wed Mar 14 18:41:23 2018 | lowest loss so far: inf (run -1)

iterations: 1.0
{'batch_size': 128,
 'init': 'he_uniform',
 'loss': 'mean_absolute_error',
 'n_layers': 3,
 'optimizer': 'adagrad',
 'scaler': 'MinMaxScaler',
 'shuffle': True}
layer 1 | size:  14 | extras: dropout
- rate: 42.6%
layer 2 | size:  10 | extras: batchnorm
layer 3 | size:  17 | extras: dropout
- rate: 19.4%
Train on 143256 samples, validate on 9011 samples
Epoch 1/1
143256/143256 [==============================] - 3s 24us/step - loss: 5.1001 - val_loss: 7.4478

# training | RMSE: 7.9557, MAE: 6.8415, R2: 0.1424, EVS: 0.7018

# test | RMSE: 8.6622, MAE: 7.4478, R2: -0.1112, EVS: 0.6635

5 seconds.

2 | Wed Mar 14 18:41:29 2018 | lowest loss so far: 8.6622 (run 1)

iterations: 1.0
{'batch_size': 128,
 'init': 'glorot_normal',
 'loss': 'mean_squared_error',
 'n_layers': 2,
 'optimizer': 'rmsprop',
 'scaler': 'StandardScale

Train on 143256 samples, validate on 9011 samples
Epoch 1/1
143256/143256 [==============================] - 16s 112us/step - loss: 39.6381 - val_loss: 18.5864

# training | RMSE: 4.7144, MAE: 3.8066, R2: 0.6988, EVS: 0.7271

# test | RMSE: 4.3112, MAE: 3.5989, R2: 0.7248, EVS: 0.7810

25 seconds.

15 | Wed Mar 14 18:44:01 2018 | lowest loss so far: 2.2635 (run 7)

iterations: 1.0
{'batch_size': 16,
 'init': 'he_uniform',
 'loss': 'mean_absolute_error',
 'n_layers': 2,
 'optimizer': 'adadelta',
 'scaler': 'MaxAbsScaler',
 'shuffle': False}
layer 1 | size:   6 | extras: None
layer 2 | size:  19 | extras: None
Train on 143256 samples, validate on 9011 samples
Epoch 1/1
143256/143256 [==============================] - 16s 114us/step - loss: 2.1737 - val_loss: 1.5298

# training | RMSE: 3.4752, MAE: 1.9142, R2: 0.8364, EVS: 0.8455

# test | RMSE: 2.2288, MAE: 1.5298, R2: 0.9264, EVS: 0.9289

24 seconds.

16 | Wed Mar 14 18:44:26 2018 | lowest loss so far: 2.2288 (run 15)

iterations: 1.0
{

 'loss': 'mean_squared_error',
 'n_layers': 2,
 'optimizer': 'adagrad',
 'scaler': None,
 'shuffle': True}
layer 1 | size:   3 | extras: batchnorm
layer 2 | size:  17 | extras: batchnorm
Train on 143256 samples, validate on 9011 samples
Epoch 1/1
143256/143256 [==============================] - 6s 42us/step - loss: 41.5421 - val_loss: 5.3538

# training | RMSE: 3.4342, MAE: 1.8550, R2: 0.8402, EVS: 0.8504

# test | RMSE: 2.3138, MAE: 1.5854, R2: 0.9207, EVS: 0.9233

10 seconds.

28 | Wed Mar 14 18:47:20 2018 | lowest loss so far: 2.2288 (run 15)

iterations: 1.0
{'batch_size': 16,
 'init': 'normal',
 'loss': 'mean_absolute_error',
 'n_layers': 4,
 'optimizer': 'adam',
 'scaler': 'MinMaxScaler',
 'shuffle': True}
layer 1 | size:   9 | extras: dropout
- rate: 35.8%
layer 2 | size:  17 | extras: dropout
- rate: 26.1%
layer 3 | size:   4 | extras: None
layer 4 | size:  20 | extras: dropout
- rate: 16.4%
Train on 143256 samples, validate on 9011 samples
Epoch 1/1
143256/143256 [============

Train on 143256 samples, validate on 9011 samples
Epoch 1/1
143256/143256 [==============================] - 5s 35us/step - loss: 72.5316 - val_loss: 13.8122

# training | RMSE: 4.1969, MAE: 3.3229, R2: 0.7613, EVS: 0.8405

# test | RMSE: 3.7165, MAE: 3.2016, R2: 0.7955, EVS: 0.9155

8 seconds.

41 | Wed Mar 14 18:51:36 2018 | lowest loss so far: 2.2288 (run 15)

iterations: 1.0
{'batch_size': 64,
 'init': 'normal',
 'loss': 'mean_absolute_error',
 'n_layers': 1,
 'optimizer': 'adagrad',
 'scaler': 'MaxAbsScaler',
 'shuffle': True}
layer 1 | size:  19 | extras: None
Train on 143256 samples, validate on 9011 samples
Epoch 1/1
143256/143256 [==============================] - 7s 46us/step - loss: 5.8897 - val_loss: 4.0995

# training | RMSE: 5.7555, MAE: 4.5870, R2: 0.5511, EVS: 0.5706

# test | RMSE: 5.0586, MAE: 4.0995, R2: 0.6210, EVS: 0.6333

10 seconds.

42 | Wed Mar 14 18:51:46 2018 | lowest loss so far: 2.2288 (run 15)

iterations: 1.0
{'batch_size': 128,
 'init': 'glorot_normal',


Train on 143256 samples, validate on 9011 samples
Epoch 1/1
143256/143256 [==============================] - 6s 43us/step - loss: 32.0038 - val_loss: 5.7067

# training | RMSE: 3.3093, MAE: 1.8319, R2: 0.8516, EVS: 0.8520

# test | RMSE: 2.3889, MAE: 1.6637, R2: 0.9155, EVS: 0.9211

9 seconds.

54 | Wed Mar 14 18:54:34 2018 | lowest loss so far: 2.1363 (run 45)

iterations: 1.0
{'batch_size': 64,
 'init': 'he_uniform',
 'loss': 'mean_squared_error',
 'n_layers': 1,
 'optimizer': 'adadelta',
 'scaler': None,
 'shuffle': True}
layer 1 | size:  10 | extras: None
Train on 143256 samples, validate on 9011 samples
Epoch 1/1
143256/143256 [==============================] - 8s 53us/step - loss: 26.5801 - val_loss: 7.0911

# training | RMSE: 3.5722, MAE: 2.2795, R2: 0.8271, EVS: 0.8603

# test | RMSE: 2.6629, MAE: 2.0091, R2: 0.8950, EVS: 0.9138

12 seconds.

55 | Wed Mar 14 18:54:46 2018 | lowest loss so far: 2.1363 (run 45)

iterations: 1.0
{'batch_size': 16,
 'init': 'he_uniform',
 'loss': '

...

143256/143256 [==============================] - 26s 180us/step - loss: 9.4019 - val_loss: 5.6877
Epoch 4/9
143256/143256 [==============================] - 25s 177us/step - loss: 9.2960 - val_loss: 6.0308
Epoch 5/9
143256/143256 [==============================] - 27s 185us/step - loss: 9.2572 - val_loss: 5.5981
Epoch 6/9
143256/143256 [==============================] - 25s 172us/step - loss: 9.2297 - val_loss: 5.8272
Epoch 7/9
143256/143256 [==============================] - 25s 175us/step - loss: 9.1933 - val_loss: 5.4136
Epoch 8/9
143256/143256 [==============================] - 26s 178us/step - loss: 9.1643 - val_loss: 5.5463
Epoch 9/9
143256/143256 [==============================] - 25s 178us/step - loss: 9.1442 - val_loss: 5.0129

# training | RMSE: 3.0033, MAE: 1.6151, R2: 0.8778, EVS: 0.8787

# test | RMSE: 2.2390, MAE: 1.4368, R2: 0.9258, EVS: 0.9260

256 seconds.

170 | Wed Mar 14 21:43:21 2018 | lowest loss so far: 2.1363 (run 45)

iterations: 9.0
{'batch_size': 64,
 'init':

Train on 143256 samples, validate on 9011 samples
Epoch 1/9
143256/143256 [==============================] - 20s 141us/step - loss: 35.4234 - val_loss: 5.5472
Epoch 2/9
143256/143256 [==============================] - 13s 94us/step - loss: 9.8711 - val_loss: 5.4963
Epoch 3/9
143256/143256 [==============================] - 14s 97us/step - loss: 9.4719 - val_loss: 5.0976
Epoch 4/9
143256/143256 [==============================] - 14s 98us/step - loss: 9.2642 - val_loss: 5.5635
Epoch 5/9
143256/143256 [==============================] - 13s 94us/step - loss: 9.1110 - val_loss: 5.5096
Epoch 6/9
143256/143256 [==============================] - 14s 96us/step - loss: 9.0410 - val_loss: 5.6722
Epoch 7/9
143256/143256 [==============================] - 14s 96us/step - loss: 8.9781 - val_loss: 5.9625
Epoch 8/9
143256/143256 [==============================] - 14s 99us/step - loss: 8.9148 - val_loss: 5.8118

# training | RMSE: 2.8916, MAE: 1.6665, R2: 0.8867, EVS: 0.8868

# test | RMSE: 2.4108, MAE

Train on 143256 samples, validate on 9011 samples
Epoch 1/9
143256/143256 [==============================] - 20s 139us/step - loss: 3.3363 - val_loss: 2.1534
Epoch 2/9
143256/143256 [==============================] - 13s 93us/step - loss: 1.9680 - val_loss: 2.5307
Epoch 3/9
143256/143256 [==============================] - 13s 93us/step - loss: 1.9504 - val_loss: 1.9008
Epoch 4/9
143256/143256 [==============================] - 13s 92us/step - loss: 1.9262 - val_loss: 2.3795
Epoch 5/9
143256/143256 [==============================] - 13s 93us/step - loss: 1.9104 - val_loss: 1.5185
Epoch 6/9
143256/143256 [==============================] - 13s 92us/step - loss: 1.8843 - val_loss: 1.4001
Epoch 7/9
143256/143256 [==============================] - 13s 93us/step - loss: 1.8607 - val_loss: 1.5993
Epoch 8/9
143256/143256 [==============================] - 13s 93us/step - loss: 1.8782 - val_loss: 1.6342
Epoch 9/9
143256/143256 [==============================] - 14s 96us/step - loss: 1.8582 - val

Train on 143256 samples, validate on 9011 samples
Epoch 1/27
143256/143256 [==============================] - 32s 225us/step - loss: 3.8314 - val_loss: 1.4422
Epoch 2/27
143256/143256 [==============================] - 25s 174us/step - loss: 2.3239 - val_loss: 1.5297
Epoch 3/27
143256/143256 [==============================] - 25s 177us/step - loss: 2.1886 - val_loss: 1.5223
Epoch 4/27
143256/143256 [==============================] - 25s 178us/step - loss: 2.1172 - val_loss: 1.5558
Epoch 5/27
143256/143256 [==============================] - 26s 181us/step - loss: 2.0948 - val_loss: 1.4894
Epoch 6/27
143256/143256 [==============================] - 26s 178us/step - loss: 2.0721 - val_loss: 1.4887

# training | RMSE: 3.3348, MAE: 1.7703, R2: 0.8493, EVS: 0.8503

# test | RMSE: 2.2315, MAE: 1.4887, R2: 0.9263, EVS: 0.9267

177 seconds.

191 | Wed Mar 14 23:19:19 2018 | lowest loss so far: 2.1363 (run 45)

iterations: 27.0
{'batch_size': 16,
 'init': 'he_uniform',
 'loss': 'mean_squared_err

Train on 143256 samples, validate on 9011 samples
Epoch 1/27
143256/143256 [==============================] - 38s 262us/step - loss: 4.0935 - val_loss: 10.1499
Epoch 2/27
143256/143256 [==============================] - 30s 211us/step - loss: 2.6048 - val_loss: 9.8023
Epoch 3/27
143256/143256 [==============================] - 31s 214us/step - loss: 2.3333 - val_loss: 9.6369
Epoch 4/27
143256/143256 [==============================] - 30s 209us/step - loss: 2.2054 - val_loss: 9.1738
Epoch 5/27
143256/143256 [==============================] - 30s 210us/step - loss: 2.1500 - val_loss: 9.8224
Epoch 6/27
143256/143256 [==============================] - 30s 210us/step - loss: 2.0998 - val_loss: 9.7007
Epoch 7/27
143256/143256 [==============================] - 30s 208us/step - loss: 2.0585 - val_loss: 10.1321
Epoch 8/27
143256/143256 [==============================] - 30s 206us/step - loss: 2.0393 - val_loss: 10.3234
Epoch 9/27
143256/143256 [==============================] - 29s 202us/step 

In [20]:
inits, layers, batches, optimizers, shuffles, scalers, losses = [], [], [], [], [], [], []
for r in sorted( results, key = lambda x: x['loss'] )[:21]:
    print("loss: {:.4} | {} seconds | {:.1f} iterations | run {} ".format( 
        r['loss'], r['seconds'], r['iterations'], r['counter'] ))
    pprint( r['params'] )
    
    inits.append( r['params']['init'])
    layers.append( r['params']['n_layers'])
    batches.append( r['params']['batch_size'])
    optimizers.append( r['params']['optimizer'])
    shuffles.append( r['params']['shuffle'])
    scalers.append( r['params']['scaler'])
    losses.append( r['params']['loss'])

loss: 2.136 | 21 seconds | 1.0 iterations | run 45 
{'batch_size': 32,
 'init': 'normal',
 'layer_1_extras': {'name': 'dropout', 'rate': 0.2509763156692161},
 'layer_1_size': 12,
 'layer_2_extras': {'name': 'dropout', 'rate': 0.26175932757538256},
 'layer_2_size': 10,
 'layer_3_extras': {'name': None},
 'layer_3_size': 15,
 'layer_4_extras': {'name': 'dropout', 'rate': 0.4441696367658411},
 'layer_4_size': 3,
 'loss': 'mean_absolute_error',
 'n_layers': 3,
 'optimizer': 'adagrad',
 'scaler': 'RobustScaler',
 'shuffle': False}
loss: 2.143 | 261 seconds | 27.0 iterations | run 168 
{'batch_size': 64,
 'init': 'glorot_uniform',
 'layer_1_extras': {'name': None},
 'layer_1_size': 18,
 'layer_2_extras': {'name': None},
 'layer_2_size': 13,
 'layer_3_extras': {'name': 'dropout', 'rate': 0.17182922040152637},
 'layer_3_size': 4,
 'layer_4_extras': {'name': 'dropout', 'rate': 0.20101357355932917},
 'layer_4_size': 10,
 'loss': 'mean_absolute_error',
 'n_layers': 2,
 'optimizer': 'adam',
 'scal

In [27]:
print(losses)

['mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_squared_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_absolute_error', 'mean_squared_error', 'mean_squared_error']


In [24]:
from statistics import mode
print('Initializer: {}\nLayers: {}\nBatch Size: {}\nOptimizer: {}\nShuffle: {}\nScaler: {}\nLoss : {}'.format(
mode(inits), mode(layers), mode(batches), mode(optimizers), mode(shuffles), mode(scalers), mode(losses)))

Initializer: normal
Layers: 2
Batch Size: 64
Optimizer: adamax
Shuffle: True
Scaler: RobustScaler
Loss : mean_absolute_error
